# Step 12: Bridge — IDE 연동 시스템

## 학습 목표

- **양방향 WebSocket 통신**으로 IDE/웹 UI와 에이전트를 연결하는 방법을 이해합니다
- **메시지 프로토콜**: user, assistant, control_request, control_response 타입을 배웁니다
- **Echo dedup 패턴**과 원격 권한 처리 방식을 구현합니다

> **Bridge란?**
> 에이전트(서버)와 IDE/웹 UI(클라이언트) 사이의 통신 계층입니다.
> Claude Code는 VS Code, JetBrains 등과 WebSocket으로 연결되어
> 실시간으로 메시지를 주고받습니다.
> 핵심 패턴: **환경 등록 -> 작업 폴링 -> 세션 생성 -> 메시지 교환 -> 종료**

---

## Claude Code 소스 분석

### 12-1. replBridge — 메인 폴링 루프 (src/bridge/replBridge.ts)

Claude Code의 Bridge는 WebSocket과 SSE(Server-Sent Events) 두 가지 전송 계층을 지원합니다.
핵심은 "폴링 루프"로, 작업 요청을 기다렸다가 세션을 생성하여 처리합니다:

```typescript
// src/bridge/replBridge.ts (핵심 흐름 간소화)

async function startBridge(config: BridgeConfig) {
  // 1. 환경 등록 — "이 에이전트가 여기 있습니다"
  await registerEnvironment(config)

  // 2. 전송 계층 초기화 (WebSocket + SSE)
  const transport = createTransport(config)

  // 3. ★ 메인 폴링 루프
  while (true) {
    // 작업이 올 때까지 대기
    const work = await transport.pollForWork()
    if (!work) continue

    // 세션 생성
    const session = createSession(work)

    // 메시지 교환 루프
    await handleSession(session, transport)

    // 세션 종료
    await session.teardown()
  }
}
```

### 12-2. 메시지 라우팅과 Echo Dedup (src/bridge/bridgeMessaging.ts)

Bridge의 가장 까다로운 문제는 **메시지 중복**입니다. WebSocket의 특성상 같은 메시지가 여러 번 도착할 수 있습니다:

```typescript
// src/bridge/bridgeMessaging.ts (핵심 간소화)

class BridgeMessaging {
  private sentIds = new Set<string>()    // ★ 이미 보낸 메시지 ID 추적

  async send(message: BridgeMessage) {
    // ★ Echo Dedup — 이미 보낸 메시지는 다시 보내지 않음
    if (this.sentIds.has(message.requestId)) {
      return  // 중복 전송 방지
    }
    this.sentIds.add(message.requestId)

    // 메모리 관리: ID가 너무 많아지면 정리
    if (this.sentIds.size > 1000) {
      this.sentIds.clear()
    }

    await this.transport.send(message.toJson())
  }

  async handleIncoming(raw: string) {
    const msg = BridgeMessage.fromJson(raw)

    // 메시지 타입별 라우팅
    switch (msg.type) {
      case 'user':
        await this.onUserMessage?.(msg)          // IDE에서 온 입력
        break
      case 'control_response':
        this.pendingControls.get(msg.requestId)   // 권한 응답 매칭
          ?.resolve(msg)
        break
    }
  }
}
```

### 12-3. 메시지 타입 (src/bridge/types.ts)

```typescript
// src/bridge/types.ts

type BridgeMessageType =
  | 'user'              // IDE → 에이전트: 사용자 입력
  | 'assistant'         // 에이전트 → IDE: 어시스턴트 응답
  | 'control_request'   // 에이전트 → IDE: "이 도구를 실행해도 될까요?"
  | 'control_response'  // IDE → 에이전트: "허용" 또는 "거부"

interface BridgeConfig {
  host: string
  port: number
  sessionId: string
}

interface SessionHandle {
  id: string
  send: (msg: BridgeMessage) => Promise<void>
  receive: () => AsyncGenerator<BridgeMessage>
  teardown: () => Promise<void>
}
```

**설계 포인트:**
- `control_request`/`control_response`로 원격 권한 확인이 가능합니다. IDE에서 사용자가 직접 "허용/거부"를 결정할 수 있습니다.
- 모든 메시지에 `request_id`가 있어 요청-응답 매칭이 가능합니다.
- Echo dedup으로 네트워크 불안정 시 메시지 중복을 방지합니다.

---

## Python으로 구현하기

### 구현 1: BridgeMessage — 메시지 프로토콜

`mini_claude/bridge/server.py`에 구현된 메시지 타입과 직렬화를 살펴봅니다.

In [ ]:
import sys, os
sys.path.insert(0, ".")

from mini_claude.bridge.server import BridgeMessage, BridgeServer
import json

# --- 1. BridgeMessage 4가지 타입 ---
print("=== BridgeMessage 타입 시연 ===\n")

# user: IDE에서 에이전트로
user_msg = BridgeMessage(
    type="user",
    content="이 프로젝트의 구조를 설명해주세요",
)
print(f"[user]             {user_msg.to_json()}")

# assistant: 에이전트에서 IDE로
assistant_msg = BridgeMessage(
    type="assistant",
    content="이 프로젝트는 mini_claude라는 에이전트 시스템입니다.",
    request_id="resp-001",
    metadata={"tokens": 42},
)
print(f"[assistant]        {assistant_msg.to_json()}")

# control_request: 에이전트가 IDE에 권한 확인
control_req = BridgeMessage(
    type="control_request",
    content="Bash 도구로 'rm -rf /tmp/test' 실행을 허용하시겠습니까?",
    request_id="ctrl-001",
    metadata={"tool": "Bash", "command": "rm -rf /tmp/test"},
)
print(f"[control_request]  {control_req.to_json()}")

# control_response: IDE가 에이전트에 허용/거부 응답
control_resp = BridgeMessage(
    type="control_response",
    content="allow",
    request_id="ctrl-001",  # request_id로 매칭
)
print(f"[control_response] {control_resp.to_json()}")

# --- 2. JSON 직렬화/역직렬화 ---
print("\n=== 직렬화 왕복 테스트 ===")
original = control_req
serialized = original.to_json()
restored = BridgeMessage.from_json(serialized)
print(f"  원본:    type={original.type}, request_id={original.request_id}")
print(f"  복원:    type={restored.type}, request_id={restored.request_id}")
print(f"  일치:    {original.type == restored.type and original.request_id == restored.request_id}")

### 구현 2: BridgeServer와 Echo Dedup

`BridgeServer`는 WebSocket 기반 양방향 통신 서버입니다.
Echo dedup 메커니즘으로 메시지 중복 전송을 방지합니다.

In [ ]:
# --- BridgeServer 구조와 Echo Dedup 메커니즘 시연 ---

# BridgeServer는 WebSocket 서버이므로 실제 실행 대신 구조를 살펴봅니다.
server = BridgeServer(host="localhost", port=8765)

print("=== BridgeServer 구조 ===")
print(f"  host:          {server.host}")
print(f"  port:          {server.port}")
print(f"  is_connected:  {server.is_connected}")  # 아직 클라이언트 없음

# --- Echo Dedup 메커니즘 시뮬레이션 ---
# 실제 BridgeServer._send()에 구현된 중복 방지 로직을 직접 시뮬레이션합니다.

print("\n=== Echo Dedup 시뮬레이션 ===")
print("메시지 중복 전송을 방지하는 핵심 패턴입니다.\n")

sent_ids: set[str] = set()

def simulate_send(msg: BridgeMessage) -> bool:
    """Echo dedup이 적용된 전송 시뮬레이션"""
    msg_id = msg.request_id
    if msg_id and msg_id in sent_ids:
        print(f"  [차단] request_id={msg_id} — 이미 전송된 메시지 (중복 방지)")
        return False
    if msg_id:
        sent_ids.add(msg_id)
    print(f"  [전송] request_id={msg_id} — type={msg.type}, content={msg.content[:40]}")
    return True

# 같은 메시지를 3번 보내려고 시도
msg = BridgeMessage(type="assistant", content="분석 결과입니다.", request_id="dedup-test-001")

print("같은 메시지를 3번 전송 시도:")
simulate_send(msg)  # 1번째: 전송됨
simulate_send(msg)  # 2번째: 차단됨 (중복)
simulate_send(msg)  # 3번째: 차단됨 (중복)

print(f"\n전송된 고유 메시지 수: {len(sent_ids)}")

# --- 메모리 관리 ---
print("\n=== 메모리 관리 ===")
print("sent_ids가 1000개를 초과하면 정리합니다.")
print("이것은 장시간 실행되는 세션에서 메모리 누수를 방지하는 패턴입니다.")

# --- 메시지 흐름 다이어그램 ---
print("\n=== 전체 메시지 흐름 ===")
print("""
  IDE (클라이언트)                    에이전트 (서버)
       │                                    │
       │──── user message ────────────────>│
       │     "파일 목록을 보여줘"              │
       │                                    │  에이전트 루프 실행
       │<─── assistant message ────────────│
       │     "파일을 읽겠습니다"                │
       │                                    │  도구 실행 전 권한 확인
       │<─── control_request ──────────────│
       │     "Bash('ls') 실행을 허용?"         │
       │                                    │
       │──── control_response ────────────>│
       │     "allow"                         │
       │                                    │  도구 실행
       │<─── assistant message ────────────│
       │     "파일 목록: ..."                  │
       │                                    │
""")

---

## 실습: Mock 클라이언트로 메시지 흐름 시뮬레이션

실제 WebSocket 연결 없이도 Bridge의 메시지 흐름을 시뮬레이션할 수 있습니다.
핸들러 등록, 메시지 수신 처리, 제어 요청 패턴을 직접 확인합니다.

In [ ]:
import asyncio

# --- Mock Bridge 시뮬레이션 ---
# WebSocket 없이 BridgeServer의 메시지 처리 로직을 시연합니다.

class MockBridgeSession:
    """WebSocket 없이 Bridge 메시지 흐름을 시뮬레이션"""

    def __init__(self):
        self.message_log: list[dict] = []
        self.server = BridgeServer()
        self._setup_handlers()

    def _setup_handlers(self):
        """핸들러 등록 — IDE에서 오는 메시지를 처리"""
        async def handle_user(msg: BridgeMessage):
            self.message_log.append({"direction": "IDE->Agent", "msg": msg})
            print(f"  [수신] user: {msg.content}")

            # 에이전트가 응답 생성 (시뮬레이션)
            response = BridgeMessage(
                type="assistant",
                content=f"'{msg.content}'에 대한 응답입니다.",
                request_id=f"resp-{len(self.message_log)}",
            )
            self.message_log.append({"direction": "Agent->IDE", "msg": response})
            print(f"  [전송] assistant: {response.content}")

        self.server.on_user_message = handle_user

    async def simulate_user_input(self, text: str):
        """사용자 입력을 시뮬레이션"""
        raw = BridgeMessage(type="user", content=text).to_json()
        await self.server._handle_raw_message(raw)

    async def simulate_control_flow(self, tool_name: str, command: str):
        """제어 요청/응답 흐름을 시뮬레이션"""
        print(f"\n  --- 제어 흐름 시뮬레이션: {tool_name}('{command}') ---")

        # 1. 에이전트가 권한 요청
        request = BridgeMessage(
            type="control_request",
            content=f"{tool_name} 도구로 '{command}' 실행을 허용하시겠습니까?",
            request_id="ctrl-sim-001",
            metadata={"tool": tool_name, "command": command},
        )
        self.message_log.append({"direction": "Agent->IDE", "msg": request})
        print(f"  [전송] control_request: {request.content}")

        # 2. IDE가 허용 응답 (시뮬레이션)
        response = BridgeMessage(
            type="control_response",
            content="allow",
            request_id="ctrl-sim-001",
        )
        self.message_log.append({"direction": "IDE->Agent", "msg": response})
        print(f"  [수신] control_response: {response.content}")

        return response.content == "allow"


# --- 실행 ---
session = MockBridgeSession()

print("=== 시나리오 1: 일반 대화 ===")
await session.simulate_user_input("현재 디렉토리의 파일을 보여주세요")

print("\n=== 시나리오 2: 제어 요청 (권한 확인) ===")
allowed = await session.simulate_control_flow("Bash", "ls -la /")
print(f"  허용 여부: {allowed}")

print("\n=== 시나리오 3: 잘못된 메시지 처리 ===")
await session.server._handle_raw_message("이것은 유효한 JSON이 아닙니다")
print("  (파싱 실패 메시지가 위에 출력됩니다)")

print(f"\n=== 전체 메시지 로그 ({len(session.message_log)}개) ===")
for i, entry in enumerate(session.message_log):
    msg = entry["msg"]
    print(f"  {i+1}. [{entry['direction']:12s}] {msg.type:18s} | {msg.content[:50]}")

---

## 핵심 설계 원칙 정리

### 1. 단순한 메시지 프로토콜 — 4가지 타입이면 충분하다
user, assistant, control_request, control_response. 이 4가지 메시지 타입만으로 에이전트와 IDE 간의 모든 상호작용을 표현할 수 있습니다. 프로토콜이 단순할수록 구현이 쉽고 디버깅이 편합니다.

### 2. Echo Dedup — 멱등성 보장
네트워크 불안정으로 같은 메시지가 여러 번 도착할 수 있습니다. `request_id` 기반 중복 제거로 한 번만 처리되도록 보장합니다. 이것은 분산 시스템에서 널리 사용되는 "at-most-once delivery" 패턴입니다.

### 3. 요청-응답 매칭 — request_id로 비동기 추적
`control_request`를 보내고 `control_response`를 기다릴 때, `request_id`로 어떤 요청에 대한 응답인지 매칭합니다. `asyncio.Future`를 사용하면 비동기적으로 깔끔하게 구현할 수 있습니다.

### 4. 전송 계층 추상화
WebSocket이든 SSE이든, 메시지 처리 로직은 동일합니다. 전송 계층을 추상화하면 새로운 프로토콜을 추가할 때 메시지 로직을 수정할 필요가 없습니다.

---

## 다음 Step 미리보기

지금까지 13개의 서브시스템을 하나씩 구현했습니다. 이제 모든 것을 하나로 합칠 시간입니다.

**Step 13**에서는:
- **13개 서브시스템을 하나의 Agent 클래스로 통합**하는 방법
- **YAML 설정 파일**로 에이전트를 구성하는 방법
- **전체 파이프라인 데이터 흐름**을 처음부터 끝까지 추적
- **워크샵 전체 요약**: 7가지 핵심 설계 원칙

을 다룹니다.